# Lecture: SAC from stable-baselines3

**Objective**: Apply Soft Actor-Critic (SAC) implementation of
`stable-baselines3` (SB3) to the `Pendulum-v1` environment already used for
DDPG and TD3 in `90-Pendulum-DDPG.ipynb`, and learn to **read SAC's training
metrics**:

- how the maximum-entropy objective changes what the actor optimizes for,
- what the automatically tuned entropy coefficient `ent_coef` does and why
  it removes a fragile hyperparameter,
- and how SAC's built-in exploration compares to DDPG/TD3's external action
  noise.

If you have not done so yet, work through `90-Pendulum-DDPG.ipynb` first:
SAC reuses the same environment, the same `MetricLogger`/`plot_metrics`
pattern, and TD3's twin-critic trick.

Run the following cell **only on Google Colab** to install
`stable-baselines3`. If you work locally and already installed it (manually or
via `requirements.txt`), skip this cell.

In [ ]:
!pip install stable-baselines3==2.6.0

### Exercise 1: Train SAC, log the metrics and plot them

The cell below trains SB3's SAC on `Pendulum-v1` with the default
**`MlpPolicy`**. As in `90-Pendulum-DDPG.ipynb`, all hyperparameters that
matter for training time and stability are collected in one `SAC_CONFIG`
dict. With `verbose=1`, SAC prints a table every few episodes; the
`MetricLogger` callback additionally records the key metrics so we can plot
them afterwards with `matplotlib` &mdash; this works identically **locally
and on Colab**, no external tools needed.

The default config below converges on Pendulum in roughly 5 minutes on a
Colab CPU/GPU runtime.

**Task**: While training runs, compare the printed table to the DDPG/TD3
table from ch9. Which metrics are new? What does `ent_coef` do, and why does
SAC not need an explicit `action_noise` entry?

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy


class MetricLogger(BaseCallback):
    """Collect selected metrics from SB3's logger during training."""

    def __init__(self, log_every=200):
        super().__init__()
        self.log_every = log_every
        self.records = {"timesteps": [], "ep_rew_mean": [], "actor_loss": [],
                        "critic_loss": [], "ent_coef": []}

    def _on_step(self) -> bool:
        if self.num_timesteps % self.log_every == 0:
            logs = self.logger.name_to_value
            ep_buffer = self.model.ep_info_buffer
            ep_rew = np.mean([e["r"] for e in ep_buffer]) if len(ep_buffer) > 0 else np.nan
            self.records["timesteps"].append(self.num_timesteps)
            self.records["ep_rew_mean"].append(ep_rew)
            self.records["actor_loss"].append(logs.get("train/actor_loss", np.nan))
            self.records["critic_loss"].append(logs.get("train/critic_loss", np.nan))
            self.records["ent_coef"].append(logs.get("train/ent_coef", np.nan))
        return True


def plot_metrics(records, title_suffix="", color="tab:blue"):
    panels = [
        ("ep_rew_mean", "Mean episodic reward", 0, "target -> 0"),
        ("actor_loss", "actor_loss", None, "should decrease"),
        ("critic_loss", "critic_loss", None, "should trend down, stay bounded"),
        ("ent_coef", "ent_coef (entropy temperature)", None, "auto-tuned, usually decays"),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for ax, (key, title, ref, note) in zip(axes, panels):
        ax.plot(records["timesteps"], records[key], color=color)
        if ref is not None:
            ax.axhline(ref, color="grey", ls="--", alpha=0.7, label=note)
            ax.legend(fontsize=8)
        ax.set_title(title + title_suffix)
        ax.set_xlabel("timesteps")
        ax.grid(True, ls="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


# All hyperparameters that affect training time and stability, in one place.
# Same speed lever as DDPG/TD3: gradient_steps caps SGD updates per train() call.
TOTAL_TIMESTEPS = 25_000
SAC_CONFIG = dict(
    learning_rate=3e-4,     # shared by actor, critics and the entropy coefficient
    buffer_size=1_000_000,  # replay buffer capacity (transitions)
    learning_starts=100,    # random-action warmup steps before training starts
    batch_size=256,         # minibatch size sampled from the replay buffer
    tau=0.005,              # polyak averaging coefficient for target networks
    gamma=0.99,             # discount factor
    train_freq=(1, "episode"),  # collect data for this long between train() calls
    gradient_steps=64,      # number of SGD updates per train() call
    ent_coef="auto",        # entropy temperature: "auto" tunes it via gradient descent
    target_entropy="auto",  # target entropy for the auto-tuning; "auto" = -dim(action space)
)

# Create the environment
env = gym.make("Pendulum-v1")

# Create a SAC agent with an MLP policy. tensorboard_log is optional (see below).
logger_cb = MetricLogger()
model = SAC("MlpPolicy", env, verbose=1, tensorboard_log="./sac_pendulum_tb/", **SAC_CONFIG)

# Train the model
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=logger_cb)

# Evaluate the trained policy over 10 episodes
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"\nMean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")

# Save the model
model.save("sac_pendulum")

plot_metrics(logger_cb.records)

### Reference: what the SAC training metrics mean

On **Pendulum-v1** the reward per step lies in $[-16.27, 0]$, so episodic
return over 200 steps is at best close to **0** and typically **-1500 to
-200** for a well-trained agent, same as for DDPG/TD3.

#### `train/` &mdash; the optimisation itself (the important diagnostics)

| Metric | Meaning | Normal / healthy range | What it tells you |
|--------|---------|--------------------------|--------------------|
| `actor_loss` | Actor loss under the maximum-entropy objective, $\mathbb E_s[\alpha \log \pi_\theta(a\|s) - Q_\phi(s,a)]$ | should **decrease**, noisier than DDPG/TD3 because of the stochastic policy | Flat/increasing &rarr; actor not improving |
| `critic_loss` | TD-error MSE, averaged over **two** critics (SAC uses TD3's clipped double-Q by default) | should **trend down**, bounded | Exploding &rarr; still possible but much rarer than DDPG |
| `ent_coef` | The entropy temperature $\alpha$ &mdash; how much the actor is rewarded for staying stochastic | starts at **1.0**, usually **decays** as the policy sharpens | Stuck high &rarr; policy stays too random; collapses to ~0 too fast &rarr; premature determinism |
| `ent_coef_loss` | Gradient-descent loss used to auto-tune `ent_coef` towards `target_entropy` | noisy around 0 | Only relevant if you set `ent_coef` to a fixed float instead of `"auto"` |

**What is different from DDPG/TD3 (see `90-Pendulum-DDPG.ipynb`), and why:**
- the actor is **stochastic** ($\pi_\theta(a\|s)$, a squashed Gaussian), not
  deterministic &mdash; so SAC explores **natively** through its own policy
  distribution and needs **no external `action_noise`**
- the objective adds an entropy bonus,
  $J(\theta) = \mathbb E\left[\sum_t r_t + \alpha\, \mathcal H(\pi_\theta(\cdot\|s_t))\right]$
  &mdash; the actor is rewarded for staying random, not just for maximizing
  reward, which is what makes SAC more robust to hyperparameters than DDPG
  (ch9 slides: "SAC robust, DDPG brittle")
- like TD3, SAC trains **two critics** and takes their minimum in the TD
  target &mdash; the overestimation fix from TD3 is baked into SAC by default,
  it is not something you additionally enable

#### `SAC_CONFIG` &mdash; what's new compared to `DDPG_CONFIG` / `TD3_CONFIG`

| Key | Meaning | Effect |
|-----|---------|--------|
| `ent_coef` | Entropy temperature $\alpha$ | `"auto"` tunes it automatically via gradient descent (recommended); a fixed float disables auto-tuning and becomes a sensitive hyperparameter again |
| `target_entropy` | Target entropy the auto-tuning drives $\alpha$ towards | `"auto"` sets it to $-\dim(\mathcal A)$, a solid default; rarely needs changing |

All other keys (`learning_rate`, `buffer_size`, `learning_starts`,
`batch_size`, `tau`, `gamma`, `train_freq`, `gradient_steps`) have the exact
same meaning as in `DDPG_CONFIG` &mdash; see the reference table in
`90-Pendulum-DDPG.ipynb` for details.

**Rules of thumb for a healthy SAC run:**
- `ep_rew_mean` rises steadily and then plateaus, typically faster and more
  reliably than DDPG.
- `ent_coef` starts high and decays smoothly &mdash; a sudden collapse to
  near-zero signals the policy became deterministic too early.
- `critic_loss` trends down and stays bounded, same as TD3.

#### Optional: TensorBoard

Because the training cell passes `tensorboard_log="./sac_pendulum_tb/"`, you
can also inspect the curves in TensorBoard.

- **Locally**: run `tensorboard --logdir ./sac_pendulum_tb/` in a terminal and
  open the printed URL.
- **On Colab**: run the two lines below in a cell.

```python
%load_ext tensorboard
%tensorboard --logdir ./sac_pendulum_tb/
```

The inline matplotlib plots above already cover the essentials, so TensorBoard
is optional here.

### Exercise 2: Fixing the entropy temperature

`ent_coef="auto"` is what makes SAC robust: the actor always has *just
enough* incentive to keep exploring, tuned automatically for the current
environment. To see why this matters, fix `ent_coef` to a small constant
instead &mdash; this removes the auto-tuning and reintroduces a sensitive
hyperparameter, similar in spirit to DDPG's external `action_noise`.

**Task**: run the cell below (same timestep budget as Exercise 1), then
compare `ep_rew_mean` to the healthy run above. With too little entropy
incentive, does the policy converge to a worse plateau, or does it get stuck
early? (There is no single "broken" curve here, unlike DDPG's divergence in
`90-Pendulum-DDPG.ipynb` &mdash; SAC's failure mode under a bad `ent_coef` is
underexploration, not blow-up.)

In [ ]:
env = gym.make("Pendulum-v1")

# Fixed, small entropy coefficient -> little incentive to keep exploring.
fixed_ent_config = SAC_CONFIG | dict(ent_coef=0.001)

fixed_ent_logger_cb = MetricLogger()
fixed_ent_model = SAC("MlpPolicy", env, verbose=0, **fixed_ent_config)
fixed_ent_model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=fixed_ent_logger_cb)

plot_metrics(fixed_ent_logger_cb.records, title_suffix=" (fixed ent_coef=0.001)", color="tab:orange")

### Exercise 3: Watch a trained episode

Test the (well-trained, `model` from Exercise 1) agent on one Pendulum
episode.

#### Run this if you use a local Python setup:

In [ ]:
import gymnasium as gym
from stable_baselines3 import SAC

env = gym.make("Pendulum-v1", render_mode="human")

# Load the model and run one episode
model = SAC.load("sac_pendulum")

obs, _ = env.reset()
done = False
while not done:
    action, _ = model.predict(obs)
    obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    env.render()

env.close()

#### Run this if you use Colab (renders to a video):

In [ ]:
import gymnasium as gym
from stable_baselines3 import SAC

from IPython.display import HTML
from base64 import b64encode

import imageio

# instantiation of the environment
env = gym.make("Pendulum-v1", render_mode="rgb_array")

# resetting the environment for first start
obs, _ = env.reset()

# initialize a list of frames for video creation
frames = []

done = False
while not done:
    # capture the frame and append it to frames list
    frame = env.render()
    frames.append(frame)

    action, _ = model.predict(obs)
    # do one step in the environment
    obs, reward, terminated, truncated, info = env.step(action)

    # flag whether the episode is finished
    done = terminated or truncated

    # final rendering for last image of episode
    if done:
      frame = env.render()
      frames.append(frame)

env.close()

# save video as
video_path = "./Pendulum_vid_own_policy.mp4"
imageio.mimsave(video_path, frames, fps=30)

In [ ]:
# this is for displaying the video after saving
mp4 = open(video_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=400 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")